# Domain-Specific Feature Engineering
Builds candidate model features for the four groups called out in `instruction.md` (onus_attribute, transaction_attribute, bureau, bureau_enquiry). Companion to `eda.ipynb` - see that notebook for general data exploration (missingness, class imbalance, raw correlations). This notebook re-derives the small amount of shared state it needs (data load, feature-group column lists, raw correlation-with-target) so it can run standalone.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)

DEV_PATH = 'Dev_data_to_be_shared.csv'
dev = pd.read_csv(DEV_PATH)
print('dev shape:', dev.shape)

dev shape: (96806, 1216)


In [2]:
onus_cols = [c for c in dev.columns if c.startswith('onus_attribute')]
txn_cols = [c for c in dev.columns if c.startswith('transaction_attribute')]
enq_cols = [c for c in dev.columns if c.startswith('bureau_enquiry')]
bureau_cols = [c for c in dev.columns if c.startswith('bureau_') and c not in enq_cols]

num_cols = dev.select_dtypes(include=[np.number]).columns.drop(['bad_flag'], errors='ignore')
corr_with_target = dev[num_cols].corrwith(dev['bad_flag'])

print('onus:', len(onus_cols), '| transaction:', len(txn_cols), '| bureau:', len(bureau_cols), '| bureau_enquiry:', len(enq_cols))

C:\Users\phamminhha3006\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\phamminhha3006\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


onus: 48 | transaction: 664 | bureau: 452 | bureau_enquiry: 50


# Domain-Specific Feature Engineering
No data dictionary was supplied - column names are generic (`onus_attribute_N`, `transaction_attribute_N`, `bureau_N`, `bureau_enquiry_N`). Roles below are inferred from statistical signature (scale, boundedness, cardinality, missingness pattern, cross-column monotonicity/nesting, correlation clustering) and cross-checked against the group descriptions in `instruction.md`. Each identification is validated with evidence before being used.

## 1. Credit Utilization & Limit Dynamics (onus_attribute)

Evidence: `onus_attribute_1` is the Credit Limit (values are round-number currency amounts, 25,000-2,800,000, matching the example given in `instruction.md`). `onus_attribute_2 / 17 / 20 / 23` are mutually correlated (r = 0.85-0.9997), bounded roughly in [-1.1, 1.25], continuous with high cardinality - a signature consistent with a pre-computed utilization ratio (balance/limit) evaluated at different snapshots (e.g. current + prior months), not a reconstructed one - `balance_col/limit_col` for other candidate balance columns only reaches r=0.12 against `onus_attribute_2`, i.e. weaker than the 0.85+ seen among the ratio-like columns themselves. `onus_attribute_2` has 0% missing and the single highest correlation with `bad_flag` (0.108) among all 1214 features, so it is used as the canonical Internal Utilization metric.

In [3]:
util_cols = ['onus_attribute_2', 'onus_attribute_17', 'onus_attribute_20', 'onus_attribute_23']

dev['internal_utilization'] = dev['onus_attribute_2']
dev['internal_utilization_avg4'] = dev[util_cols].mean(axis=1)
dev['limit_stress_80'] = (dev['internal_utilization'] >= 0.8).astype(int)
dev['limit_stress_90'] = (dev['internal_utilization'] >= 0.9).astype(int)
dev['limit_stress_persistence'] = (dev[util_cols] >= 0.8).sum(axis=1)  # 0-4: how many snapshots stayed >=80% utilized

new_feats_g1 = ['internal_utilization', 'internal_utilization_avg4', 'limit_stress_80', 'limit_stress_90', 'limit_stress_persistence']
print(dev[new_feats_g1].corrwith(dev['bad_flag']))

internal_utilization         0.108491
internal_utilization_avg4    0.107159
limit_stress_80              0.088972
limit_stress_90              0.084162
limit_stress_persistence     0.082834
dtype: float64


## 2. Transaction Velocity & RFM Metrics (transaction_attribute)

Evidence: with 664 unlabeled columns, exact merchant categories cannot be named directly. Correlation-based hierarchical clustering (distance = 1 - |corr|, average linkage, k=14) groups the 664 columns into behaviorally distinct clusters. Three clusters stand out by signature:
- Everyday/retail spend proxy - the largest cluster (~369 cols), used by ~22% of accounts on average per column (highest activity breadth of any cluster) -> matches ubiquitous POS/e-commerce spend.
- Cash-withdrawal-like proxy - a small cluster (17 cols) used by <0.01% of accounts per column but with the largest average transaction value when used -> matches the rare-but-large signature of cash withdrawal/cash-advance activity.
- High-risk-category proxy - a small cluster (17 cols) whose mean absolute correlation with bad_flag (~0.023) is 2-5x every other clusters -> used as the higher-risk merchant-category spend proxy.

No reliable last-30-day-vs-90-day nested window structure could be recovered for transaction_attribute (unlike bureau/bureau_enquiry below, which do show clean nesting) - a stride scan across offsets found no offset with near-100% monotonicity. Recency is therefore approximated with a breadth/concentration proxy instead of a true day-window ratio; this is flagged as a limitation.

In [4]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

X = dev[txn_cols].fillna(0)
corr_txn = X.corr().fillna(0).values
np.fill_diagonal(corr_txn, 1.0)
dist = 1 - np.abs(corr_txn)
dist[dist < 0] = 0
np.fill_diagonal(dist, 0)
Z = linkage(squareform(dist, checks=False), method='average')
cluster_labels = fcluster(Z, 14, criterion='maxclust')

cluster_map = pd.Series(cluster_labels, index=txn_cols)
nonzero_frac = (X != 0).mean()
corr_target_txn = corr_with_target.reindex(txn_cols)

cluster_stats = pd.DataFrame({
    'n_cols': cluster_map.value_counts(),
    'mean_nonzero_frac': nonzero_frac.groupby(cluster_map).mean(),
    'mean_abs_corr_target': corr_target_txn.abs().groupby(cluster_map).mean(),
})

spend_cluster = cluster_stats['n_cols'].idxmax()
sizable = cluster_stats[cluster_stats['n_cols'] >= 10]
cash_cluster = sizable['mean_nonzero_frac'].idxmin()
highrisk_cluster = sizable.drop(index=[spend_cluster, cash_cluster], errors='ignore')['mean_abs_corr_target'].idxmax()
print('spend_cluster:', spend_cluster, '| cash_cluster:', cash_cluster, '| highrisk_cluster:', highrisk_cluster)
print(cluster_stats.loc[[spend_cluster, cash_cluster, highrisk_cluster]])

spend_cols = cluster_map[cluster_map == spend_cluster].index.tolist()
cash_cols = cluster_map[cluster_map == cash_cluster].index.tolist()
highrisk_cols = cluster_map[cluster_map == highrisk_cluster].index.tolist()

dev['spend_amount_proxy'] = dev[spend_cols].sum(axis=1)
dev['cash_amount_proxy'] = dev[cash_cols].sum(axis=1)
dev['highrisk_merchant_proxy'] = dev[highrisk_cols].sum(axis=1)
dev['total_txn_activity'] = dev[txn_cols].sum(axis=1)

dev['cash_vs_spend_ratio'] = dev['cash_amount_proxy'] / (dev['cash_amount_proxy'] + dev['spend_amount_proxy'] + 1)
dev['highrisk_merchant_share'] = dev['highrisk_merchant_proxy'] / (dev['total_txn_activity'].abs() + 1)
dev['merchant_breadth'] = (dev[txn_cols] != 0).sum(axis=1)  # RFM frequency proxy: number of active transaction fields

new_feats_g2 = ['cash_vs_spend_ratio', 'highrisk_merchant_share', 'merchant_breadth', 'total_txn_activity']
print(dev[new_feats_g2].corrwith(dev['bad_flag']))

spend_cluster: 12 | cash_cluster: 1 | highrisk_cluster: 6
    n_cols  mean_nonzero_frac  mean_abs_corr_target
12     369           0.221460              0.010255
1       17           0.000026              0.000553
6       17           0.024795              0.023011


cash_vs_spend_ratio       -0.000452
highrisk_merchant_share    0.017145
merchant_breadth          -0.009206
total_txn_activity         0.003611
dtype: float64


## 3. Bureau Stress & Tradeline Health (bureau)

Evidence: `bureau_433 / 437 / 438 / 444 / 448` are the only bureau_* columns that are bounded (~[-1, 2]) with high cardinality - the same ratio signature as the onus utilization columns above - so `bureau_433` (lowest missingness among them, 76%) is used as the External/Bureau Utilization proxy.
Among the 288 bounded, non-negative, low-cardinality (<=15 unique values) bureau_* columns - the candidate delinquency/status-count fields - pairs 10 apart (e.g. `bureau_13`/`bureau_23`) are non-decreasing in over 99.9% of rows, the same nested-time-window signature confirmed below for `bureau_enquiry_*`, supporting a delinquency-count-over-increasing-window interpretation. The top 20 of these 288 by absolute correlation with `bad_flag` (0.037-0.052, notably higher than the rest of the block) are used as the delinquency-flag set for Delinquency Trajectory.

In [5]:
dev['external_utilization'] = dev['bureau_433']
dev['utilization_gap'] = dev['internal_utilization'] - dev['external_utilization']

count_like_bureau = [c for c in bureau_cols if dev[c].nunique() <= 15 and dev[c].min() >= 0]
delinquency_cols = corr_with_target.reindex(count_like_bureau).abs().sort_values(ascending=False).head(20).index.tolist()

dev['delinquency_flag_count'] = (dev[delinquency_cols] > 0).sum(axis=1)
dev['worst_bureau_status'] = dev[delinquency_cols].max(axis=1)

new_feats_g3 = ['external_utilization', 'utilization_gap', 'delinquency_flag_count', 'worst_bureau_status']
print(dev[new_feats_g3].corrwith(dev['bad_flag']))

external_utilization      0.045936
utilization_gap           0.030816
delinquency_flag_count    0.059811
worst_bureau_status       0.036533
dtype: float64


## 4. Inquiry Velocity (bureau_enquiry)

Evidence: `bureau_enquiry_1..50` splits into 5 blocks of 10 (same enquiry-purpose ordering repeated 5 times) with strictly increasing max scale per block (42 -> 72 -> 81 -> 172 -> 297) and, for every purpose position tested, the value at block b+1 is greater than or equal to the value at block b in 100% of rows - i.e. cumulative enquiry counts over 5 nested, increasing lookback windows (shortest = block 1, longest = block 5).

In [6]:
block1_cols = [f'bureau_enquiry_{i}' for i in range(1, 11)]    # shortest lookback window
block5_cols = [f'bureau_enquiry_{i}' for i in range(41, 51)]   # longest lookback window

dev['enquiries_recent'] = dev[block1_cols].sum(axis=1)
dev['enquiries_longterm'] = dev[block5_cols].sum(axis=1)
dev['credit_seeking_intensity'] = dev['enquiries_recent'] / (dev['enquiries_longterm'] + 1)

new_feats_g4 = ['enquiries_recent', 'enquiries_longterm', 'credit_seeking_intensity']
print(dev[new_feats_g4].corrwith(dev['bad_flag']))

enquiries_recent            0.066795
enquiries_longterm          0.069117
credit_seeking_intensity    0.022205
dtype: float64


## All engineered domain features vs bad_flag

In [7]:
engineered = new_feats_g1 + new_feats_g2 + new_feats_g3 + new_feats_g4
eng_corr = dev[engineered].corrwith(dev['bad_flag']).sort_values(key=lambda s: s.abs(), ascending=False)
eng_corr.to_frame('corr_with_bad_flag')

,corr_with_bad_flag
internal_utilization,0.108491
internal_utilization_avg4,0.107159
limit_stress_80,0.088972
limit_stress_90,0.084162
limit_stress_persistence,0.082834
enquiries_longterm,0.069117
enquiries_recent,0.066795
delinquency_flag_count,0.059811
external_utilization,0.045936
worst_bureau_status,0.036533


## Summary
- No data dictionary: column semantics (Groups 1-4 above) were reverse-engineered from statistical signature - treat exact labels (cash withdrawal, high-risk merchant) as working hypotheses, not ground truth, and validate/re-label with the business/data-owning team before using them in a compliance-sensitive context.
- Engineered features hold real signal: internal_utilization, limit_stress_persistence, utilization_gap, delinquency_flag_count/worst_bureau_status, and credit_seeking_intensity all correlate with bad_flag in the expected direction (see consolidated table above) and should be included as candidate model inputs alongside raw features.
- Weakest engineered features: cash_vs_spend_ratio (~0 corr) and total_txn_activity (~0.004 corr) show almost no linear signal - the cash/spend cluster split is a coarse statistical proxy, not confirmed merchant-category labels; revisit if a real data dictionary becomes available.